# TD : RAG 

Ce TD est divisé en plusieurs parties et traite de la construction d'un RAG :
1. Récupèration d'un corpus
2. Différente méthode de chunking
3. Embeddings, vecteurs et similarité 
4. Ingestion avec Qdrant
5. Retrieval 
6. Reranking

* Corpus [BEIR](https://huggingface.co/datasets/BeIR/hotpotqa?utm_source=chatgpt.com)
* Corpus [Narrativeqa-RAG](https://huggingface.co/datasets/DinoStackAI/narrativeqa-rag?utm_source=chatgpt.com)
* Vector DB [Qdrant](https://qdrant.tech/documentation/quickstart/)


Pour démarrer Qdrant:

```bash
docker pull qdrant/qdrant
```


```bash
docker run -p 6333:6333 -p 6334:6334 \
    -v "$(pwd)/qdrant_storage:/qdrant/storage:z" \
    qdrant/qdrant
```
Web UI : http://localhost:6333/dashboard#/welcome

In [ ]:
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt

## 1. Chargez un corpus et garder un sous-ensemble 

Ici on charge le corpus Narrativeqa-RAG via HF (api datasets), et on garde un sous-ensemble de texte pour la suite

In [ ]:
## instancier le corpus Narrativeqa-RAG et examiner sont contenu. 

corpus  = load_dataset(
    "DinoStackAI/narrativeqa-rag",
    "corpus",
    split="train",
    streaming=True,
)

In [ ]:
## Afficher quelques instances, leur id, leur titre, et le text associés 
n_instances = ... 
for doc in corpus.take(n_instances):
    ...

In [ ]:
### Garder un subset pour la construction du RAG

N_docs = 500
SEED = 42 # pour la reproductibilité 

shuffle_corpus = corpus.shuffle(
    seed=SEED, 
    buffer_size=10000)

small_corpus = []

for doc in shuffle_corpus.take(N_docs):
    small_corpus.append({
        "doc_id":doc["id"], 
        "title":doc["title"],
        "text":doc["text"]
    })

## 2. Chunking

Dans cette section on va voir différente méthode de chunking afin de séparer un texte en chunk (morceaux de texte). On va voir 3 stratégies : 
1. Taille fixe 
2. Avec recouvrement
3. Par phrase (ou paragraphe)


In [ ]:
## Représentation graphique de la taille de chaque texte  : 

doc_lengths = [
    len(doc["text"].split())
    for doc in small_corpus
]

print(f"Nombre de documents : {len(doc_lengths)}")
print(f"Moyenne : {np.mean(doc_lengths):.1f} mots")
print(f"Médiane : {np.median(doc_lengths):.1f} mots")
print(f"Minimum : {np.min(doc_lengths)} mots")
print(f"Maximum : {np.max(doc_lengths)} mots")

plt.hist(doc_lengths, bins=30)
plt.xlabel("Nombre de mots par document")
plt.ylabel("Nombre de documents")
plt.title("Distribution des longueurs du corpus")
plt.show()

1. Chunk par taille fixe 

Définir un fonction qui prend en input un texte et un nombre de mots, et renvoie une liste de chunks de taille fixe 

In [ ]:
def chunk_fixed(text, chunk_size=150):
    ...


In [ ]:
text = small_corpus[0]["text"]

chunks = chunk_fixed(text, chunk_size=100)

print(*chunks, sep="\n\n")

2. Chunk avec overlap 

Définir un fonction qui prend en input un texte, un nombre de mots et un overlap (le n derniers mots d'un chunk sont les n premiers du chunk suivant), et renvoie une liste de chunks

In [ ]:
def chunk_overlap(text, chunk_size=150, overlap=20):
    ...

In [ ]:
chunks_overlap = chunk_overlap(text, chunk_size=100)

print(*chunks_overlap, sep="\n\n")

3. Chunk par phrase 

Définir un fonction qui prend en input un texte, de phrase et renvoie une liste de chunks

In [ ]:
def chunk_sentence(text, chunk_size=1): # chunk_size = nb de phrase
    ...

In [ ]:
chunks_sentence = chunk_sentence(text)

print(*chunks_sentence, sep="\n\n")

4. Corpus et chunks 

Il est temps de constituer le corpus de chunks. Constituer l'un ensemble des chunks à partir du small_corpus, en gardant les infos sur le texte originale (id, title) et la place du chunk dans le texte. 

On pourra tester les différentes méthodes ainsi que la méthode sans chunk. 

In [ ]:
cor_chunks = []
...

cor_chunks_fixed = []  
...

cor_chunks_overlap= []
...

cor_chunks_sentence = []


In [ ]:
print("no chunking size:", len(cor_chunks))

print("Fixed chunking size:", len(cor_chunks_fixed))
print("overlape chunking size:", len(cor_chunks_overlap))
print("Sentence chunking size:", len(cor_chunks_sentence))

## 3. Embedding, vecteur et similarité 

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

In [ ]:
embedding_model = SentenceTransformer(...)

In [ ]:

print("Model:", MODEL_NAME)
print("Embedding dimension:", embedding_model.get_embedding_dimension())
print("Max sequence length:", embedding_model.max_seq_length, "tokens")

In [ ]:
## Pour encoder une phrase :
embedding = embedding_model.encode(text)
print(embedding)

Encoder les chunks du corpus (choisir la méthode de chunking). Construire un corpus avec, pour chaque instance, *doc_id*, *title*, *chunk*, *chunk_id* et *embedding*

In [ ]:
embedding_corpus = []

for i, instance in enumerate(small_corpus):
...

In [ ]:
embedding_corpus[:10]

## 4. Vector DB et ingestion 

Avec Qdrant :
- créer une collection 
- stocker les embeddings et les payloads 

In [ ]:
from qdrant_client import QdrantClient, models
from qdrant_client.models import PointStruct

In [ ]:
client = QdrantClient(url="http://localhost:6333")
print(client.get_collections())

In [ ]:
## Créer une collection 

COLLECTION_NAME = "TD_6_collection"
VECTOR_SIZE = embedding_corpus[0]["embedding"].shape[0]

print(VECTOR_SIZE)

In [ ]:
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)
    print(f"Collection {COLLECTION_NAME} supprimée.")
    
res = client.create_collection(
    collection_name=COLLECTION_NAME, 
    vectors_config=models.VectorParams(
        size=VECTOR_SIZE, 
        distance=models.Distance.COSINE ## Distance utiliser pour le retrieval
    )
)

if res:
    print(f"Collection {COLLECTION_NAME} créée.")

In [ ]:
## ajouter les embeddings à la collection
## 1. Créer les points


points = []

for i, instance in enumerate(embedding_corpus):
    points.append(
        PointStruct(
            id=i,  ### identifiant du chunk dans la vector DB, on pourra ausis utiser un UUID dans des plus grosses DB 
            vector=instance["embedding"].tolist(), 
            payload={
                "doc_id":instance["doc_id"], 
                "chunk_id": instance["chunk_id"], 
                "title":instance["title"], 
                "chunk":instance["chunk"],
                #"text":instance["text"]
            }
        )
    )


In [ ]:
## 2. Ajouter les points à la collection

client.upsert(collection_name=COLLECTION_NAME, points=points)

print(f"Ingestion des {len(embedding_corpus)} points dans la collection {COLLECTION_NAME} terminée")

In [ ]:
collection_info = client.get_collection(COLLECTION_NAME)

print("Points dans Qdrant :", collection_info.points_count)
print("Points préparés :", len(points))

## 5. Retrieval 

1. Se connecter au client 
2. Vérifier que la collection existe 
3. Définir une fonction de **retrieve** et effectuer une recherche
4. Charger une passe de questions 

In [ ]:
## 1. Se connecter au client 

client = QdrantClient(url="http://localhost:6333")

## 2. Vériier que la collection existe
COLLECTION_NAME = "TD_6_collection"
#print(client.get_collections())
if client.collection_exists(COLLECTION_NAME):
    print(f"La collection '{COLLECTION_NAME}' existe")
else:
    print(f"La collection '{COLLECTION_NAME}' n'existe pas")


In [ ]:
## 3. Définir une fonction de retrieval

def retrieval(query, client, collection_name, embedding_model, top_k=5):
    # 1. Encoder la question
    query_embedding = embedding_model.encode(
        query
    )
    
    # 2. Recherche les chunks dans Qdrant
    response = client.query_points(
        collection_name=collection_name, 
        query=query_embedding.tolist(), 
        limit=top_k, 
        with_payload=True
    )
    
    return [
        {
            "rank": rank,
            "score": point.score,
            "chunk_id": point.payload.get("chunk_id"),
            "doc_id": point.payload.get("doc_id"),
            "title": point.payload.get("title"),
            "chunk":point.payload.get("chunk")
        }
        for rank, point in enumerate(response.points, start=1)
    ]  

In [ ]:
## Effectuer une recherche : 

query = "What causes the seasons on Earth?"

hits = retrieval(
    query=query,
    client=client,
    collection_name=COLLECTION_NAME,
    embedding_model=embedding_model,
    top_k=5
)

In [ ]:
import pandas as pd

df_hits = pd.DataFrame([
    {
        "rank": hit["rank"],
        "score": round(hit["score"], 4),
        "title": hit["title"],
        "doc_id": hit["doc_id"],
        "chunk": hit["chunk"]
    }
    for hit in hits
])

df_hits